In [2]:
import os, random
from dataclasses import dataclass
from typing import List, Dict, Tuple

import pandas as pd
from tqdm import tqdm

from rdkit import Chem
from rdkit import RDLogger

import torch

RDLogger.DisableLog("rdApp.*")


In [3]:
@dataclass
class CFG:
    url: str = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/qm9.csv"
    smiles_col: str = "smiles"
    seed: int = 42

    # Splits
    train_frac: float = 0.90
    val_frac: float = 0.05 

    # Tokenization
    max_len: int = 80  

    # Caching
    cache_dir: str = "./cache_qm9"
    cache_file: str = "qm9_processed.pt"

cfg = CFG()

PAD, BOS, EOS = "<PAD>", "<BOS>", "<EOS>"

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)

set_seed(cfg.seed)
os.makedirs(cfg.cache_dir, exist_ok=True)
cache_path = os.path.join(cfg.cache_dir, cfg.cache_file)

print("Cache path:", cache_path)


Cache path: ./cache_qm9/qm9_processed.pt


In [4]:
def canonicalize_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)  # canonical SMILES

# quick sanity example
print(canonicalize_smiles("C(C)O"))   # ethanol-ish representation
print(canonicalize_smiles("NOT_A_SMILES"))

CCO
None


In [5]:
if os.path.exists(cache_path):
    print("Loading cached dataset:", cache_path)
    obj = torch.load(cache_path)
    train_sm, val_sm, test_sm = obj["splits"]
else:
    print("Downloading QM9 CSV...")
    df = pd.read_csv(cfg.url)
    raw_smiles = df[cfg.smiles_col].astype(str).tolist()
    print("Raw rows:", len(raw_smiles))

    print("Canonicalizing + filtering invalid SMILES...")
    smiles = []
    for s in tqdm(raw_smiles):
        cs = canonicalize_smiles(s)
        if cs is not None:
            smiles.append(cs)

    print("Valid SMILES:", len(smiles))

    # reproducible shuffle
    random.shuffle(smiles)

    n = len(smiles)
    n_train = int(cfg.train_frac * n)
    n_val = int(cfg.val_frac * n)

    train_sm = smiles[:n_train]
    val_sm = smiles[n_train:n_train + n_val]
    test_sm = smiles[n_train + n_val:]

    torch.save({"splits": (train_sm, val_sm, test_sm), "cfg": cfg.__dict__}, cache_path)
    print("Saved cache:", cache_path)

print("Split sizes:", len(train_sm), len(val_sm), len(test_sm))


Raw rows: 133885
Canonicalizing + filtering invalid SMILES...


100%|██████████| 133885/133885 [00:15<00:00, 8786.06it/s] 


Valid SMILES: 133885
Saved cache: ./cache_qm9/qm9_processed.pt
Split sizes: 120496 6694 6695


In [6]:
def build_char_vocab(smiles_list: List[str]) -> Tuple[Dict[str, int], List[str]]:
    chars = set()
    for s in smiles_list:
        chars.update(list(s))
    itos = [PAD, BOS, EOS] + sorted(chars)  # index -> token
    stoi = {c: i for i, c in enumerate(itos)}  # token -> index
    return stoi, itos

stoi, itos = build_char_vocab(train_sm)

print("Vocab size:", len(itos))
print("First 20 vocab tokens:", itos[:20])
print("Last 10 vocab tokens:", itos[-10:])


Vocab size: 24
First 20 vocab tokens: ['<PAD>', '<BOS>', '<EOS>', '#', '(', ')', '+', '-', '1', '2', '3', '4', '5', '=', 'C', 'F', 'H', 'N', 'O', '[']
Last 10 vocab tokens: ['C', 'F', 'H', 'N', 'O', '[', ']', 'c', 'n', 'o']


In [7]:
def encode(sm: str, stoi: Dict[str, int], max_len: int) -> List[int]:
    pad_id = stoi[PAD]
    bos_id = stoi[BOS]
    eos_id = stoi[EOS]

    ids = [bos_id] + [stoi[c] for c in sm] + [eos_id]

    # pad or truncate
    if len(ids) < max_len:
        ids = ids + [pad_id] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
        ids[-1] = eos_id  # ensure EOS exists
    return ids

def decode(ids: List[int], itos: List[str]) -> str:
    stoi_local = {c: i for i, c in enumerate(itos)}
    pad_id = stoi_local[PAD]
    bos_id = stoi_local[BOS]
    eos_id = stoi_local[EOS]

    out = []
    for i in ids:
        if i == eos_id:
            break
        if i in (pad_id, bos_id):
            continue
        out.append(itos[i])
    return "".join(out)


In [8]:
for i in range(5):
    s = train_sm[i]
    ids = encode(s, stoi, cfg.max_len)
    s2 = decode(ids, itos)
    print("\nOriginal:", s)
    print("Decoded :", s2)
    print("Equal?  :", s == s2)



Original: CN=C(C#N)OCC=O
Decoded : CN=C(C#N)OCC=O
Equal?  : True

Original: CC#CCC1NC1C
Decoded : CC#CCC1NC1C
Equal?  : True

Original: CC12CC(C=O)C1(C)N2
Decoded : CC12CC(C=O)C1(C)N2
Equal?  : True

Original: Cn1cnc(N)nc1=O
Decoded : Cn1cnc(N)nc1=O
Equal?  : True

Original: C#CC1=CC(C)CC1=O
Decoded : C#CC1=CC(C)CC1=O
Equal?  : True


In [9]:
def length_stats(smiles_list: List[str]):
    lens = sorted([len(s) for s in smiles_list])
    n = len(lens)

    def pct(p):
        return lens[int(p * (n - 1))]

    return {
        "count": n,
        "min": min(lens),
        "p50": pct(0.50),
        "p90": pct(0.90),
        "p99": pct(0.99),
        "max": max(lens),
        "avg": sum(lens) / n,
    }

stats = length_stats(train_sm)
print("Train SMILES length stats:", stats)

# sequences longer than max_len after adding BOS/EOS
truncated = sum(1 for s in train_sm if (len(s) + 2) > cfg.max_len)
print(f"Truncated (train): {truncated} / {len(train_sm)}")

if truncated > 0:
    print("Suggestion: increase cfg.max_len to 100 or 120 to reduce truncation.")


Train SMILES length stats: {'count': 120496, 'min': 1, 'p50': 15, 'p90': 18, 'p99': 22, 'max': 34, 'avg': 15.064599654760324}
Truncated (train): 0 / 120496


In [10]:
processed_path = os.path.join(cfg.cache_dir, "qm9_splits_vocab.pt")
torch.save(
    {
        "splits": (train_sm, val_sm, test_sm),
        "stoi": stoi,
        "itos": itos,
        "cfg": cfg.__dict__,
    },
    processed_path,
)

print("Saved:", processed_path)

Saved: ./cache_qm9/qm9_splits_vocab.pt
